# SatDetection — Colab Training Notebook
**Multi-task U-Net + ResNet-50 for building and road segmentation**

CSCI 4366/6366 — Neural Networks and Deep Learning, Spring 2026  
Team: Rufai Yakubu, Aaron Tyler

---
**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Check GPU

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations rasterio wandb pyyaml tqdm

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR       = '/content/drive/MyDrive/SatDetection'
DATA_DIR       = os.path.join(BASE_DIR, 'data')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
VISUALS_DIR    = os.path.join(BASE_DIR, 'outputs/visuals')

for d in [
    DATA_DIR, CHECKPOINT_DIR, VISUALS_DIR,
    os.path.join(DATA_DIR, 'landcoverai/raw'),
    os.path.join(DATA_DIR, 'landcoverai/tiles/images'),
    os.path.join(DATA_DIR, 'landcoverai/tiles/masks'),
]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'Base: {BASE_DIR}')

## 4. Clone Project Repository

In [ ]:
import os, sys

REPO_URL = 'https://github.com/YOUR_USERNAME/SatDetection.git'  # <-- update this

if not os.path.exists('/content/SatDetection'):
    !git clone $REPO_URL /content/SatDetection
else:
    !git -C /content/SatDetection pull

sys.path.insert(0, '/content/SatDetection/src')
print('Repo ready.')

## 5. W&B Login

In [ ]:
import wandb
wandb.login()

## 6. Data Preparation — Tile LandCover.ai

**One-time step.** Upload the extracted LandCover.ai zip contents to Drive at:
```
MyDrive/SatDetection/data/landcoverai/raw/
```
LandCover.ai v1 puts everything in one folder — images and masks together:
```
raw/
  M-33-20-C-c-3-4.tif       <- image
  M-33-20-C-c-3-4_m.tif     <- mask (same name + _m)
  ...
```
No need to separate them — the tiling script handles this automatically.

In [ ]:
# Inspect what's actually in the raw folder before tiling
import glob

LC_RAW_DIR   = os.path.join(DATA_DIR, 'landcoverai/raw')
LC_TILES_OUT = os.path.join(DATA_DIR, 'landcoverai/tiles')

all_tifs    = sorted(glob.glob(os.path.join(LC_RAW_DIR, '**/*.tif'), recursive=True))
image_tifs  = [f for f in all_tifs if not f.endswith('_m.tif')]
mask_tifs   = [f for f in all_tifs if f.endswith('_m.tif')]

print(f'Raw folder  : {LC_RAW_DIR}')
print(f'Total .tifs : {len(all_tifs)}')
print(f'Images      : {len(image_tifs)}')
print(f'Masks       : {len(mask_tifs)}')
if image_tifs:
    print(f'Example     : {os.path.basename(image_tifs[0])}')

In [ ]:
# Run tiling — skips if tiles already exist
from utils import tile_all

existing = len(glob.glob(os.path.join(LC_TILES_OUT, 'images/*.npy')))
if existing > 0:
    print(f'Tiles already exist: {existing}. Skipping tiling.')
else:
    assert len(image_tifs) > 0, 'No images found in raw folder. Upload LandCover.ai data first.'
    # pass same dir for both image_dir and mask_dir — tile_all finds _m.tif masks automatically
    total = tile_all(LC_RAW_DIR, LC_RAW_DIR, LC_TILES_OUT, tile_size=256, stride=256)
    print(f'Done. Total tiles: {total}')

In [ ]:
# Verify tiles
import numpy as np

n_images = len(glob.glob(os.path.join(LC_TILES_OUT, 'images/*.npy')))
n_masks  = len(glob.glob(os.path.join(LC_TILES_OUT, 'masks/*.npy')))
print(f'Image tiles : {n_images}')
print(f'Mask tiles  : {n_masks}')
assert n_images == n_masks, 'Mismatch!'
assert n_images > 0, 'No tiles found — check the raw folder and re-run tiling.'

sample_img  = np.load(sorted(glob.glob(os.path.join(LC_TILES_OUT, 'images/*.npy')))[0])
sample_mask = np.load(sorted(glob.glob(os.path.join(LC_TILES_OUT, 'masks/*.npy')))[0])
print(f'Image shape : {sample_img.shape}  dtype: {sample_img.dtype}')
print(f'Mask shape  : {sample_mask.shape}  dtype: {sample_mask.dtype}')
print(f'Mask values : building max={sample_mask[0].max():.0f}  road max={sample_mask[1].max():.0f}')

## 7. Training Config

In [ ]:
BASE_CONFIG = {
    'landcoverai_root':        os.path.join(DATA_DIR, 'landcoverai/tiles'),
    'checkpoint_dir':          CHECKPOINT_DIR,
    'visuals_dir':             VISUALS_DIR,
    'wandb_project':           'SatDetection',
    'epochs':                  50,
    'batch_size':              16,
    'num_workers':             2,
    'early_stopping_patience': 8,   # increased — random init needs more time
    'w_building':              1.0,
    'w_road':                  2.0,
}

# Model A uses lower lr — random init is more sensitive to large steps
# Model B uses standard lr — pretrained weights are already in a good region
EXPERIMENTS = [
    {'pretrained': False, 'name': 'Model A — Random Init',        'lr': 0.00005},
    {'pretrained': True,  'name': 'Model B — ImageNet Pretrained', 'lr': 0.0001},
]

print('Experiment plan:')
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f'  Run {i}: {exp["name"]} | lr={exp["lr"]}')

## 8. Training

In [ ]:
from train import train

def run_experiment(exp_index):
    exp    = EXPERIMENTS[exp_index - 1]
    config = {**BASE_CONFIG, 'pretrained': exp['pretrained'], 'lr': exp['lr']}
    print(f"\n{'='*60}")
    print(f"Run {exp_index}: {exp['name']} | lr={exp['lr']}")
    print(f"{'='*60}")
    checkpoint = train(config)
    print(f"Checkpoint: {checkpoint}")
    return checkpoint

In [ ]:
# Diagnostic — confirm tiles visible before training
tiles_path = os.path.join(DATA_DIR, 'landcoverai/tiles/images')
found = sorted(glob.glob(os.path.join(tiles_path, '*.npy')))
print(f'Tiles path : {tiles_path}')
print(f'Tiles found: {len(found)}')
if len(found) == 0:
    print('\nERROR: No tiles found. Go back and re-run Section 6.')
else:
    print('Ready to train.')

In [ ]:
# Run 1 — Model A (random init, baseline)
ckpt_1 = run_experiment(1)

In [ ]:
# Run 2 — Model B (ImageNet pretrained)
ckpt_2 = run_experiment(2)

## 9. Evaluation

In [ ]:
from evaluate import evaluate

def eval_experiment(exp_index, checkpoint_path):
    exp    = EXPERIMENTS[exp_index - 1]
    config = {**BASE_CONFIG, 'pretrained': exp['pretrained']}
    print(f"\n{'='*60}")
    print(f"Evaluating Run {exp_index}: {exp['name']}")
    print(f"{'='*60}")
    return evaluate(config, checkpoint_path, save_visuals=True, n_visuals=5)

In [ ]:
results_1 = eval_experiment(1, ckpt_1)

In [ ]:
results_2 = eval_experiment(2, ckpt_2)

## 10. Results Summary

In [ ]:
import pandas as pd

rows = []
for exp, results in [
    (EXPERIMENTS[0], results_1),
    (EXPERIMENTS[1], results_2),
]:
    rows.append({
        'Model':        exp['name'],
        'IoU Building': f"{results['iou_building']:.4f}",
        'IoU Road':     f"{results['iou_road']:.4f}",
        'Mean IoU':     f"{(results['iou_building'] + results['iou_road']) / 2:.4f}",
        'F1 Building':  f"{results['f1_building']:.4f}",
        'F1 Road':      f"{results['f1_road']:.4f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))